# Historical Baltic basin ice conditions

This notebook processes the winter chart archive one date at a time. It calculates 15% SIC extent, mapped ice thickness and chart occurrence. The record is weekly through 2005/06 and daily from 2006/07. The latest winter is partial.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except ImportError as exc:
    raise ImportError("Install geo dependencies: python -m pip install -e '.[geo]'") from exc

def project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in (p, *p.parents):
        if (q / 'pyproject.toml').exists() and (q / 'notebooks').exists(): return q
    raise FileNotFoundError('Project root not found')

ROOT = project_root(); RAW = ROOT / 'data' / 'raw'
import cartopy
cartopy.config['data_dir'] = str(ROOT / '.cache' / 'cartopy')
OUT = ROOT / 'data' / 'processed'; FIG = ROOT / 'figures' / 'generated'
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
THRESHOLD = 15.0
THRESHOLDS = (15.0,)
SPATIAL_SUPPORT = 0.80
DOMAIN_COVERAGE = 0.80
THICKNESS_COVERAGE = 0.80



def theil_sen_fit(x, y):
    """Median pairwise slope and median intercept for short annual series."""
    x=np.asarray(x,dtype=float); y=np.asarray(y,dtype=float)
    keep=np.isfinite(x)&np.isfinite(y); x=x[keep]; y=y[keep]
    if len(x)<3: return np.nan,np.nan
    i,j=np.triu_indices(len(x),k=1)
    slope=float(np.median((y[j]-y[i])/(x[j]-x[i])))
    return slope,float(np.median(y-slope*x))

def area_km2(lat, lon):
    lat = np.asarray(lat, float); lon = np.asarray(lon, float)
    def edges(x):
        mid = (x[:-1] + x[1:]) / 2
        return np.r_[x[0] - (mid[0] - x[0]), mid, x[-1] + (x[-1] - mid[-1])]
    le = np.clip(edges(lat), -90, 90); oe = edges(lon)
    south = np.minimum(le[:-1], le[1:]); north = np.maximum(le[:-1], le[1:])
    dlon = np.abs(np.deg2rad(np.diff(oe)))
    return 6371.0088**2 * dlon[None, :] * (np.sin(np.deg2rad(north))-np.sin(np.deg2rad(south)))[:, None]

sic_path, thk_path = {}, {}
for y in range(1980, 2020):
    a = (y // 5) * 5
    sic_path[y] = RAW/'historical_baltic_sic_chunks'/f'baltic_sic_{a}_{a+4}.nc'
    thk_path[y] = RAW/'historical_baltic_thickness_chunks'/f'baltic_thickness_{a}_{a+4}.nc'
for y in range(2020, 2025):
    sic_path[y] = RAW/'recent_baltic_winter_subsets'/f'baltic_ice_concentration_winter_{y}_{y+1}.nc'
    thk_path[y] = RAW/'historical_baltic_thickness_chunks'/'baltic_thickness_2020_2024.nc'
sic_path[2025] = RAW/'historical_baltic_sic_chunks'/'baltic_sic_2025_partial_to_2026-05-10.nc'
thk_path[2025] = RAW/'historical_baltic_thickness_chunks'/'baltic_thickness_2025_partial_to_2026-05-10.nc'
missing = [str(p) for m in (sic_path, thk_path) for p in m.values() if not p.is_file()]
if missing: raise FileNotFoundError('Missing data files:\n'+'\n'.join(missing))
print(f'{len(sic_path)} winter records found; source files are local.')

## Coverage and winter summaries

Missing cells remain missing. A chart date is used for winter summaries when at least 80% of the season's mapped basin area is present. Thickness is averaged over mapped ice only when at least 80% of that area has valid thickness.


In [ ]:
# Process one 2-D chart at a time. Each winter contributes equally to climatology maps.
thresholds=THRESHOLDS
sum_maps={}; n_maps={}; rows=[]; chart_rows=[]; lat=lon=cell_area=None
for threshold in thresholds:
    sum_maps[threshold]={}; n_maps[threshold]={}

for y in range(1980,2026):
    era='partial' if y==2025 else ('weekly' if y<=2005 else 'daily')
    print(f'Processing {y}/{str(y+1)[-2:]} ({era})',flush=True)
    with xr.open_dataset(sic_path[y],decode_cf=True,mask_and_scale=True) as sd, xr.open_dataset(thk_path[y],decode_cf=True,mask_and_scale=True) as td:
        sic=sd['ice_concentration']; thk=td['ice_thickness']
        if lat is None:
            lat,lon=sic.latitude.values,sic.longitude.values
            cell_area=area_km2(lat,lon).astype(np.float64)
            for threshold in thresholds:
                for group in ('all','weekly','daily'):
                    sum_maps[threshold][group]=np.zeros(cell_area.shape,np.float64)
                    n_maps[threshold][group]=np.zeros(cell_area.shape,np.uint16)
        elif not (np.array_equal(lat,sic.latitude.values) and np.array_equal(lon,sic.longitude.values)):
            raise ValueError(f'Grid changed in {sic_path[y].name}')

        dates=pd.DatetimeIndex(pd.to_datetime(sic.time.values))
        use=np.flatnonzero((((dates.year==y)&(dates.month>=11))|((dates.year==y+1)&(dates.month<=5))))
        thk_days=pd.DatetimeIndex(pd.to_datetime(thk.time.values)).normalize()
        thk_index={d:i for i,d in enumerate(thk_days)}
        valid_n=np.zeros(cell_area.shape,np.uint16)
        ice_n={threshold:np.zeros(cell_area.shape,np.uint16) for threshold in thresholds}
        day_rows=[]; n_observations=0
        for i in use:
            field=np.asarray(sic.isel(time=int(i)).values,np.float32)
            valid=np.isfinite(field)
            if not valid.any(): continue
            n_observations+=1; valid_n+=valid
            valid_area=float(cell_area[valid].sum())
            threshold_areas={}
            ice_masks={}
            for threshold in thresholds:
                ice=valid&(field>=threshold)
                ice_masks[threshold]=ice
                ice_n[threshold]+=ice
                threshold_areas[int(threshold)]=float(cell_area[ice].sum())
            ice15=ice_masks[THRESHOLD]
            sia=float(np.sum(cell_area[valid]*field[valid]/100.0))
            day=dates[i].normalize(); ti=thk_index.get(day)
            mean_thk=np.nan; thk_cover=np.nan
            if ti is not None and threshold_areas[int(THRESHOLD)]>0:
                tf=np.asarray(thk.isel(time=ti).values,np.float32)
                ok=ice15&np.isfinite(tf)&(tf>=0)
                covered=float(cell_area[ok].sum())
                thk_cover=covered/threshold_areas[int(THRESHOLD)]
                if covered>0: mean_thk=float(np.sum(cell_area[ok]*tf[ok])/covered)
            day_rows.append({
                'date':dates[i], 'valid_area_km2':valid_area,
                **{f'extent_{int(t)}_km2':threshold_areas[int(t)] for t in thresholds},
                'sea_ice_area_km2':sia,'thickness_mean_cm':mean_thk,
                'thickness_ice_area_coverage':thk_cover,
            })

        if not n_observations:
            warnings.warn(f'No valid SIC dates in winter {y}'); continue
        season_area=float(cell_area[valid_n>0].sum())
        for row in day_rows:
            row['mapped_area_coverage']=row['valid_area_km2']/season_area
        accepted=[r for r in day_rows if r['mapped_area_coverage']>=DOMAIN_COVERAGE]
        support=valid_n>=int(np.ceil(SPATIAL_SUPPORT*n_observations))
        for threshold in thresholds:
            occurrence=np.divide(ice_n[threshold],valid_n,
                out=np.full(cell_area.shape,np.nan,np.float32),where=valid_n>0)
            occurrence[~support]=np.nan
            if era in ('weekly','daily'):
                for group in ('all',era):
                    ok=np.isfinite(occurrence)
                    sum_maps[threshold][group][ok]+=occurrence[ok]
                    n_maps[threshold][group][ok]+=1

        # Keep chart-date values after the same spatial-coverage screen used for seasonal means.
        for r in accepted:
            chart_rows.append({
                'winter_start_year':y,'winter':f'{y}/{str(y+1)[-2:]}','era':era,
                'date':r['date'],'mapped_area_coverage':r['mapped_area_coverage'],
                **{f'extent_{int(t)}_km2':r[f'extent_{int(t)}_km2'] for t in thresholds},
                'sea_ice_area_km2':r['sea_ice_area_km2'],
                'thickness_ice_area_coverage':r['thickness_ice_area_coverage'],
                'thickness_mean_cm':(r['thickness_mean_cm']
                    if np.isfinite(r['thickness_mean_cm']) and
                       r['thickness_ice_area_coverage']>=THICKNESS_COVERAGE else np.nan),
            })

        extents={int(t):np.asarray([r[f'extent_{int(t)}_km2'] for r in accepted],float)
                 for t in thresholds}
        thick_rows=[r['thickness_mean_cm'] for r in accepted
                    if np.isfinite(r['thickness_mean_cm']) and
                       r['thickness_ice_area_coverage']>=THICKNESS_COVERAGE]
        calendar_coverage=n_observations/len(use) if len(use) else np.nan
        row={
            'winter_start_year':y,'winter':f'{y}/{str(y+1)[-2:]}','era':era,
            'complete_winter':y<2025,'valid_chart_dates':n_observations,
            'calendar_coordinates_in_season':len(use),
            'calendar_date_coverage_fraction':calendar_coverage,
            'dates_meeting_80pct_mapped_area':len(accepted),
            'mean_sea_ice_area_km2':float(np.mean([r['sea_ice_area_km2'] for r in accepted])) if accepted else np.nan,
            'ice_positive_observation_dates_15pct':int(np.sum(extents[15]>0)) if len(accepted) else np.nan,
            'observed_ice_positive_days_15pct':(
                int(np.sum(extents[15]>0)) if era=='daily' and y<2025 and calendar_coverage>=0.80 else np.nan),
            'valid_thickness_dates':len(thick_rows),
            'mean_ice_thickness_cm':float(np.mean(thick_rows)) if thick_rows else np.nan,
        }
        for threshold in thresholds:
            z=extents[int(threshold)]
            key=int(threshold)
            row[f'maximum_sampled_extent_{key}_km2']=float(z.max()) if z.size else np.nan
            row[f'mean_sampled_extent_{key}_km2']=float(z.mean()) if z.size else np.nan
        rows.append(row)

summary=pd.DataFrame(rows).sort_values('winter_start_year').reset_index(drop=True)
chart_timeseries=pd.DataFrame(chart_rows).sort_values('date').reset_index(drop=True)
maps={threshold:{} for threshold in thresholds}
map_vars={}
for threshold in thresholds:
    key=int(threshold)
    for era in ('all','weekly','daily'):
        maps[threshold][era]=np.divide(
            sum_maps[threshold][era],n_maps[threshold][era],
            out=np.full(cell_area.shape,np.nan,np.float32),where=n_maps[threshold][era]>0).astype(np.float32)
        map_vars[f'occurrence_{key}pct_{era}']=(("latitude","longitude"),maps[threshold][era])
        map_vars[f'winter_support_{key}pct_{era}']=(("latitude","longitude"),n_maps[threshold][era])

summary_path=OUT/'historical_baltic_winter_summary.csv'
chart_path=OUT/'historical_baltic_chart_timeseries.csv'
map_path=OUT/'historical_baltic_occurrence_climatology.nc'
summary.to_csv(summary_path,index=False)
chart_timeseries.to_csv(chart_path,index=False)
map_ds=xr.Dataset(map_vars,coords={'latitude':lat,'longitude':lon},attrs={
    'primary_sic_threshold_percent':'15',
    'metric':'equal-winter mean of per-winter valid-chart occurrence frequency',
    'weekly_era':'1980/81-2005/06','daily_era':'2006/07-2024/25',
    'per_winter_minimum_pixel_support_fraction':SPATIAL_SUPPORT,
    'source_product_id':'SEAICE_BAL_PHY_L4_MY_011_019'})
map_ds.to_netcdf(map_path)
print(summary_path); print(chart_path); print(map_path)
display(summary.tail(8))


In [ ]:
# Era maps at the primary 15% threshold, plus the between-era difference.
weekly=maps[15.0]['weekly']*100; daily=maps[15.0]['daily']*100
fig,axes=plt.subplots(1,3,figsize=(16,6),subplot_kw={'projection':ccrs.PlateCarree()},constrained_layout=True)
panels=[(weekly,'Weekly 1980/81-2005/06','viridis',0,100),
        (daily,'Daily 2006/07-2024/25','viridis',0,100),
        (daily-weekly,'Daily minus weekly (pp)','RdBu_r',-40,40)]
for ax,(z,title,cmap,vmin,vmax) in zip(axes,panels):
    im=ax.pcolormesh(lon,lat,z,transform=ccrs.PlateCarree(),cmap=cmap,vmin=vmin,vmax=vmax,shading='auto',rasterized=True)
    ax.set_extent([float(lon.min()),float(lon.max()),float(lat.min()),float(lat.max())],crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale('50m'),facecolor='#e8e5df',edgecolor='#555',linewidth=.35,zorder=2)
    ax.coastlines(resolution='50m',linewidth=.45,color='#353b3e',zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale('50m'),linewidth=.35,edgecolor='#777',zorder=3)
    gl=ax.gridlines(draw_labels=True,linewidth=.25,color='#86939a',alpha=.55,linestyle=':'); gl.top_labels=False; gl.right_labels=False
    ax.set_title(title,loc='left',fontsize=10,fontweight='semibold')
    cb=fig.colorbar(im,ax=ax,orientation='horizontal',pad=.07,shrink=.82)
    cb.set_label('Valid chart observations with SIC >= 15% (%)' if vmax==100 else 'Difference (percentage points)')
fig.suptitle('Baltic basin sea-ice chart occurrence by observation era',fontsize=14,fontweight='bold')
map_fig=FIG/'historical_baltic_occurrence_eras.png'; fig.savefig(map_fig,dpi=180,bbox_inches='tight',facecolor='white'); plt.show()
print(map_fig)

## Chart observations and winter means

Faint points show accepted chart dates; diamonds show the mean across accepted dates for each complete winter. The trend lines are fitted to winter means separately for weekly and daily charts. The partial 2025/26 season is shown as individual points only.


In [ ]:
# Raw chart-date observations plus one winter-mean marker per complete season.
chart_ts=pd.read_csv(OUT/'historical_baltic_chart_timeseries.csv',parse_dates=['date'])
complete=summary[summary.complete_winter].copy()
era_style={'weekly':('#276A8A','Weekly'),'daily':('#C85C3D','Daily')}
metric_specs=[
    (None,'SIC-derived extent at 15%','thousand km$^2$',1000.0),
    ('mean_ice_thickness_cm','Mean thickness over mapped ice','cm',1.0),
]
fig,(ax_extent,ax_thickness)=plt.subplots(2,1,figsize=(15,8),sharex=True,constrained_layout=True)
trend_rows=[]
for era,(color,label) in era_style.items():
    raw=chart_ts[chart_ts.era==era]
    ax_extent.scatter(raw.date,raw.extent_15_km2/1000,s=5,alpha=.20,color=color,label=f'{label} charts (>=80% mapped area)',rasterized=True,zorder=1)
    thick=raw[np.isfinite(raw.thickness_mean_cm)]
    ax_thickness.scatter(thick.date,thick.thickness_mean_cm,s=5,alpha=.20,color=color,label=f'{label} charts (>=80% mapped area)',rasterized=True,zorder=1)
    seasonal=complete[complete.era==era]
    center_dates=pd.to_datetime((seasonal.winter_start_year+1).astype(str)+'-02-15')
    for ax,mean_col,scale,unit in (
        (ax_extent,'mean_sampled_extent_15_km2',1000.0,'thousand km$^2$'),
        (ax_thickness,'mean_ice_thickness_cm',1.0,'cm'),
    ):
        values=seasonal[mean_col].to_numpy(dtype=float)
        years=seasonal.winter_start_year.to_numpy(dtype=float)
        slope,intercept=theil_sen_fit(years,values)
        slope_per_decade=slope*10/scale
        ax.scatter(center_dates,values/scale,s=42,marker='D',color=color,edgecolor='white',linewidth=.6,
                   label=f'{label} winter mean (trend {slope_per_decade:+.1f} {unit}/decade)',zorder=4)
        x_year=np.array([years.min(),years.max()])
        x_date=pd.to_datetime((x_year.astype(int)+1).astype(str)+'-02-15')
        ax.plot(x_date,(intercept+slope*x_year)/scale,color=color,ls='--',lw=1.8,zorder=3)
        trend_rows.append({'metric':mean_col,'era':era,'first_winter':int(years.min()),
            'last_winter':int(years.max()),'n_winters':len(years),
            'theil_sen_slope_per_decade':slope_per_decade,
            'units_per_decade':unit.replace('$^2$','^2')+'/decade'})

# Show available records from the incomplete current winter without a seasonal mean or trend.
partial=chart_ts[chart_ts.era=='partial']
if not partial.empty:
    ax_extent.scatter(partial.date,partial.extent_15_km2/1000,s=6,alpha=.25,color='#777777',label='Partial 2025/26 charts (>=80% mapped area)',rasterized=True,zorder=1)
    pt=partial[np.isfinite(partial.thickness_mean_cm)]
    ax_thickness.scatter(pt.date,pt.thickness_mean_cm,s=6,alpha=.25,color='#777777',label='Partial 2025/26 charts (>=80% mapped area)',rasterized=True,zorder=1)

for ax in (ax_extent,ax_thickness):
    ax.axvline(pd.Timestamp('2006-12-01'),color='#555',ls=':',lw=.9)
    ax.grid(axis='y',alpha=.25)
    ax.legend(frameon=False,ncol=1,fontsize=8,loc='upper left',bbox_to_anchor=(1.005,1.0),borderaxespad=0)
ax_extent.set_ylabel('thousand km$^2$')
ax_extent.set_title('Coverage-screened chart-date extent and winter mean',loc='left',fontsize=12)
ax_thickness.set_ylabel('cm')
ax_thickness.set_title('Coverage-screened chart-date thickness and winter mean',loc='left',fontsize=12)
ax_thickness.set_xlabel('Date')
ax_thickness.xaxis.set_major_locator(mdates.YearLocator(5))
ax_thickness.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.suptitle('Historical Baltic basin extent and ice thickness | chart observations and winter means',fontsize=14,fontweight='bold')
trend_summary=pd.DataFrame(trend_rows)
trend_path=OUT/'historical_baltic_trend_summary.csv'
trend_summary.to_csv(trend_path,index=False)
series_fig=FIG/'historical_baltic_extent_and_thickness.png'
fig.savefig(series_fig,dpi=180,bbox_inches='tight',facecolor='white'); plt.show()
display(trend_summary.round(2))
print('Chart-date data:',OUT/'historical_baltic_chart_timeseries.csv')
print('Trend table:',trend_path)
print('Figure:',series_fig)

## Notes

The upper series is mapped extent at the 15% SIC threshold. The lower series is area-weighted ice thickness over mapped ice, in centimetres. Trends are descriptive and are not adjusted for serial correlation. Weekly chart dates are not daily ice-day observations.

Sources: [Copernicus Marine Baltic Sea Ice product](https://data.marine.copernicus.eu/product/SEAICE_BAL_PHY_L4_MY_011_019/description), [Product User Manual](https://documentation.marine.copernicus.eu/PUM/CMEMS-SEAICE-PUM-011-019.pdf), [Quality Information Document](https://documentation.marine.copernicus.eu/QUID/CMEMS-SEAICE-QUID-011-004-011-019.pdf).
